# N150 · 快照、历史查询与可持久化

**目标：** 在保存历史版本时复用未变部分且不污染旧结果。

**先修：** N042, N005。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 按索引版本历史；二分；稀疏记录；结构共享；路径复制；持久化线段树拓展。

## 从问题到算法

每次snap复制整个数组代价很大，且多数位置没有变化。按索引记录稀疏版本历史：一次快照只生成新版本号，查询时找不晚于目标版本的最后一条修改。同一未提交版本内多次set只保留最终值。它支持单点历史查询，不自动提供高效的历史区间和。

## 最小实现

**本章接口：** `SnapshotArray.set/snap/get`

In [1]:
from bisect import bisect_right

class SnapshotArray:
    def __init__(self,length):
        if length<0: raise ValueError('nonnegative length required')
        self.history=[[(0,0)] for _ in range(length)]; self.current=0
    def _check_index(self,index):
        if not 0<=index<len(self.history): raise IndexError('index outside snapshot array')
    def set(self,index,val):
        self._check_index(index); history=self.history[index]
        if history[-1][0]==self.current: history[-1]=(self.current,val)
        else: history.append((self.current,val))
    def snap(self):
        result=self.current; self.current+=1; return result
    def get(self,index,snap_id):
        self._check_index(index)
        if not 0<=snap_id<self.current: raise ValueError('snapshot must already be committed')
        history=self.history[index]; position=bisect_right(history,snap_id,key=lambda item:item[0])-1
        return history[position][1]

## 正确性、前提与复杂度

一个位置在两次修改之间保持常值，因此历史查询等价于时间前驱查询。版本号单调，二分右界减一得到最后一个version≤snap_id的记录。同版本覆盖只影响尚未提交的状态；已提交版本记录不会被后续set覆盖。

**代价：** set均摊O(1)，snap O(1)，get O(log 该位置修改版本数)，空间O(长度+保留修改数)。持久化线段树可用路径复制支持历史区间查询，但属于拓展；本章只实现稀疏单点历史。Python要求支持bisect的key参数（3.10及以上）。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

snapshots=SnapshotArray(3); snapshots.set(0,5); first=snapshots.snap(); snapshots.set(0,6); snapshots.set(0,7); second=snapshots.snap()
show_table(['索引','稀疏版本记录','快照0值','快照1值'],[(i,h,snapshots.get(i,first),snapshots.get(i,second)) for i,h in enumerate(snapshots.history)])

索引,稀疏版本记录,快照0值,快照1值
0,"[(0, 5), (1, 7)]",5,7
1,"[(0, 0)]",0,0
2,"[(0, 0)]",0,0


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
array=SnapshotArray(3); assert array.snap()==0 and array.get(1,0)==0
array.set(0,5); array.set(0,6); sid=array.snap(); assert array.get(0,sid)==6 and array.get(0,0)==0
array.set(0,8); assert array.get(0,sid)==6
from random import Random
rng=Random(150); fast=SnapshotArray(7); values=[0]*7; copies=[]
for _ in range(900):
    op=rng.randrange(3)
    if op==0:
        i=rng.randrange(7); value=rng.randrange(-100,101); fast.set(i,value); values[i]=value
    elif op==1:
        assert fast.snap()==len(copies); copies.append(values[:])
    elif copies:
        i=rng.randrange(7); version=rng.randrange(len(copies)); assert fast.get(i,version)==copies[version][i]
for version,values in enumerate(copies):
    for i,x in enumerate(values): assert fast.get(i,version)==x
assert all(all(a[0]<b[0] for a,b in zip(history,history[1:])) for history in fast.history)
print('N150: 所有本章断言通过')

N150: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 合并同一snap前对同一索引的多次set。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 比较完整拷贝与路径复制。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1146. Snapshot Array（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。